<a href="https://colab.research.google.com/github/cylin577/Image2Audio/blob/main/i2agpu.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# @title Install dependency
!pip install gradio
!pip install soundfile
!pip install pillow
!pip install cupy-cuda12x

In [ ]:
# @title Main
import cupy as cp  # 使用 CuPy 替代 NumPy
from PIL import Image
import soundfile as sf
import gradio as gr
from scipy.io.wavfile import write

# 英文語言支持
LANGUAGE = {
    "title": "Image-Audio Encoder/Decoder",
    "encode_button": "Select Image to Encode",
    "decode_button": "Select Audio to Decode",
    "success_encode": "Image encoded to audio.",
    "success_decode": "Audio decoded back to image.",
    "error_length": "Decoded data length does not match expected length.",
}

def encode_image_to_audio(image):
    # 使用 CuPy 處理數據
    data = cp.array(image)  # CuPy 陣列
    height, width, _ = data.shape
    audio_data = data / 255.0 * 2 - 1  # 正規化數據到 [-1, 1]
    audio_data_flattened = audio_data.flatten()  # 展平數據
    size_info = cp.array([height, width], dtype=cp.int32)  # 儲存圖像大小
    audio_data_with_size = cp.concatenate([size_info, audio_data_flattened])

    audio_path = "encoded_audio.wav"
    # 確保音訊以 NumPy 格式保存，避免衝突
    write(audio_path, 44100, cp.asnumpy(audio_data_with_size).astype(cp.float32))

    return LANGUAGE["success_encode"], audio_path

def decode_audio_to_image(audio):
    # 使用 soundfile 讀取音訊數據
    audio_data, sample_rate = sf.read(audio)
    height = int(audio_data[0])  # 讀取高度
    width = int(audio_data[1])   # 讀取寬度
    data = cp.array(audio_data[2:])  # 使用 CuPy 加速數據處理
    expected_length = height * width * 3  # 應有的數據長度

    # 檢查數據長度是否正確
    if len(data) != expected_length:
        return LANGUAGE["error_length"], None

    # 恢復圖像數據
    img_data = ((data + 1) / 2 * 255).astype(cp.uint8)  # 將數據轉換回圖像
    img_data = img_data.reshape((height, width, 3))

    img = Image.fromarray(cp.asnumpy(img_data))  # 轉換為 NumPy 陣列供 PIL 使用
    image_path = "decoded_image.png"
    img.save(image_path)

    return LANGUAGE["success_decode"], image_path

# Gradio介面設置
def interface():
    with gr.Blocks() as demo:
        gr.Markdown(f"## {LANGUAGE['title']}")
        with gr.Row():
            with gr.Column():
                # 編碼部分
                img_input = gr.Image(label=LANGUAGE["encode_button"])
                audio_output = gr.File(label="Encoded Audio File")
                encode_button = gr.Button(LANGUAGE["encode_button"])

                # 綁定按鈕事件
                encode_button.click(encode_image_to_audio, inputs=[img_input], outputs=[gr.Text(label="Outputs"), audio_output])

            with gr.Column():
                # 解碼部分
                audio_input = gr.Audio(label=LANGUAGE["decode_button"], type="filepath")
                img_output = gr.Image(label="Decoded Image")
                decode_button = gr.Button(LANGUAGE["decode_button"])

                # 綁定按鈕事件
                decode_button.click(decode_audio_to_image, inputs=[audio_input], outputs=[gr.Text(label="Outputs"), img_output])

    return demo

# 運行應用，加入 `share=True` 和 `debug=True`
interface().launch(share=True, debug=True)
